# Capstone Function 6
You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. 

To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.

 Input | Output | Goal |
|-------|--------|------|
| 5D Array (20, 5) | 1D Array (20, ) | Maximise |

## Initial Submission

Bayesian Optimization for 5D cake recipe (flour, sugar, eggs, butter, milk) to maximize combined score.

### Step 1: Import Libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
print("Ready!")

### Step 2: Load & Visualize Data

In [ ]:
X_init = np.load('../../data/f6/updated_inputs - Week 4.npy')
y_init = np.load('../../data/f6/updated_outputs - Week 4.npy')
ingredients = ['Flour', 'Sugar', 'Eggs', 'Butter', 'Milk']
print(f"Loaded 5D cake recipe data: {X_init.shape[0]} samples")
print(f"Best score: {y_init.max():.6f}")

In [ ]:
# Visualize selected pairs (5D has 10 pairs - show subset)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
pairs = [(0,1), (0,2), (1,2), (2,3), (3,4), (0,4)]
for idx, (i, j) in enumerate(pairs):
    axes[idx].scatter(X_init[:, i], X_init[:, j], c=y_init, cmap='viridis', edgecolors='black', s=60)
    axes[idx].set_xlabel(ingredients[i])
    axes[idx].set_ylabel(ingredients[j])
    axes[idx].grid(True, alpha=0.3)
plt.suptitle('5D Cake Recipe - Selected Ingredient Pairs', fontweight='bold')
plt.tight_layout()
plt.show()

### Step 3: Hyperparameters

**5D Cake Recipe:**
- Restarts: 20, Raw: 2048 (higher dimensional space)
- EI with GP Matern 5/2 kernel

In [ ]:
# All inputs must be in range [0, 0.999999] per submission requirements
N_DIM = X_init.shape[1]
BOUNDS = torch.tensor([[0.0] * N_DIM, [0.999999] * N_DIM], dtype=torch.float64)
NUM_RESTARTS, RAW_SAMPLES = 20, 2048
X_train = torch.tensor(X_init, dtype=torch.float64)
y_train = torch.tensor(y_init, dtype=torch.float64).unsqueeze(-1)
gp_model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)
EI = ExpectedImprovement(gp_model, best_f=y_train.max().item())
candidate, acq_value = optimize_acqf(EI, bounds=BOUNDS, q=1, num_restarts=NUM_RESTARTS, raw_samples=RAW_SAMPLES)
next_point = candidate.detach().numpy()[0]
print(f"✓ Next recipe: {dict(zip(ingredients, next_point))}")

### Step 4: Visualize Progress & Lengthscales

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Safe lengthscale access (kernel structure may vary)
if hasattr(gp_model.covar_module, 'base_kernel'):
    ls = gp_model.covar_module.base_kernel.lengthscale.detach().numpy()[0]
else:
    ls = gp_model.covar_module.lengthscale.detach().numpy()[0]

ax1.bar(ingredients, ls, color='coral', edgecolor='black')
ax1.set_ylabel('Lengthscale')
ax1.set_title('Ingredient Importance (smaller = more sensitive)', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

best_so_far = np.maximum.accumulate(y_init)
ax2.plot(range(1, len(best_so_far)+1), best_so_far, 'b-o', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Best Score')
ax2.set_title('Optimization Progress', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Next: {next_point}")

### Visualize Expected Improvement

For higher-dimensional spaces, we visualize 1D slices of the acquisition function.
Each plot shows how EI changes along one dimension while others are fixed at the proposed point.

In [ ]:
# 1D marginal plots of Expected Improvement
n_points = 100
n_dims = len(next_point)

fig, axes = plt.subplots(1, n_dims, figsize=(4*n_dims, 4))
if n_dims == 1:
    axes = [axes]

for dim in range(n_dims):
    # Create points varying along this dimension
    X_marginal = np.tile(next_point, (n_points, 1))
    X_marginal[:, dim] = np.linspace(0, 0.999999, n_points)
    X_marginal_torch = torch.tensor(X_marginal, dtype=torch.float64)
    
    # Compute EI at each point
    with torch.no_grad():
        ei_values = EI(X_marginal_torch.unsqueeze(1)).numpy()
    
    # Plot
    axes[dim].plot(X_marginal[:, dim], ei_values, 'b-', linewidth=2)
    axes[dim].axvline(next_point[dim], color='r', linestyle='--', linewidth=2, label='Proposed')
    axes[dim].set_xlabel(f'x{dim+1}', fontsize=12)
    axes[dim].set_ylabel('Expected Improvement' if dim == 0 else '', fontsize=12)
    axes[dim].set_title(f'EI along dim {dim+1}', fontsize=11, fontweight='bold')
    axes[dim].grid(True, alpha=0.3)
    if dim == 0:
        axes[dim].legend()

plt.suptitle('Expected Improvement - 1D Marginals', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Red dashed lines show the proposed next point coordinates.")
print(f"EI is maximized when considering all dimensions jointly.")

### Step 5: Format Next Query for Submission

Format the proposed next sample point in the required submission format:
- Format: `x1-x2-x3-...-xn` where each xᵢ begins with 0
- Precision: 6 decimal places per coordinate
- Range: All values clamped to [0, 0.999999]

In [ ]:
# Format the next query for submission
def format_query(point):
    """Format point as x1-x2-...-xn with 6 decimal places, clamped to [0, 0.999999]."""
    clamped = [max(0.0, min(0.999999, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

# Clamp next_point to valid range
next_point_clamped = np.array([max(0.0, min(0.999999, x)) for x in next_point])

# Display the formatted submission query
submission_query = format_query(next_point)
print("=" * 60)
print("SUBMISSION QUERY FOR FUNCTION 6")
print("=" * 60)
print(f"\n{submission_query}\n")
print("=" * 60)
print(f"\nCoordinates breakdown:")
for i, x in enumerate(next_point, 1):
    print(f"  x{i} = {x:.6f}")
print(f"\nEI value: {acq_value.item():.6f}")
if acq_value.item() > 0.1:
    print("  -> High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  -> Moderate EI: Some exploration potential remains")
else:
    print("  -> Low EI: Approaching convergence")
print(f"Current best observed: {y_train.max().item():.6f}")

## Week 5 — Neural Network Surrogate (PyTorch)

This section replaces the Gaussian Process surrogate with a **Neural Network** model for the 5D optimization problem (f6).

**Why Neural Network for f6?**
- Neural networks can approximate arbitrarily complex functions — suited for higher-dimensional (5D) problems where GP covariance matrices become expensive
- MC Dropout provides uncertainty estimates at inference time without requiring ensemble training — each forward pass with random dropout masks produces a different prediction, and 50 passes give mean and variance
- Input gradient magnitude reveals which dimensions the network relies on most, analogous to GP lengthscales or tree feature importance
- Flexible architecture can capture non-linear interactions between input dimensions

**Architecture:** 5 -> 64 -> 32 -> 1 (fully connected, ReLU activations, Dropout p=0.2)
**Acquisition Strategy:** UCB with MC Dropout uncertainty (mean ± kappa * std from 50 stochastic forward passes)

### Step 1: Load Data & Set Up PyTorch

Load Week 5 data (25 samples), normalize inputs/outputs, and define the neural network architecture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Load Week 5 cumulative data
X_w5 = np.load('../../data/f6/updated_inputs - Week 5.npy')
y_w5 = np.load('../../data/f6/updated_outputs - Week 5.npy')

print(f"Week 5 Data: {X_w5.shape[0]} samples, {X_w5.shape[1]} dimensions")
print(f"Output range: [{y_w5.min():.6f}, {y_w5.max():.6f}]")
print(f"Best observed: {y_w5.max():.6f} at {X_w5[y_w5.argmax()]}")

# Normalize inputs and outputs for training stability
X_mean, X_std = X_w5.mean(axis=0), X_w5.std(axis=0) + 1e-8
y_mean, y_std = y_w5.mean(), y_w5.std() + 1e-8

X_norm = (X_w5 - X_mean) / X_std
y_norm = (y_w5 - y_mean) / y_std

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y_norm, dtype=torch.float32).unsqueeze(1)

# Define neural network with Dropout
class SurrogateNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model = SurrogateNN()
print(f"\nModel architecture:\n{model}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):.0f}")

### Step 2: Neural Network Hyperparameters & Training

**Hyperparameter Choices:**
1. **Hidden layers**: 64 -> 32 — progressively narrowing architecture captures hierarchical features
2. **Dropout = 0.2**: Applied after each hidden layer for MC Dropout uncertainty at inference
3. **Learning rate = 0.01**: Adam optimizer with moderate learning rate for 25 samples
4. **Epochs = 500**: Sufficient training iterations; loss curve monitored for convergence
5. **MC samples = 50**: Forward passes with dropout enabled at inference for uncertainty estimation
6. **UCB kappa = 2.0**: Exploration parameter scaled for 5D search space

In [ ]:
# Training hyperparameters
LEARNING_RATE = 0.01
EPOCHS = 500
MC_SAMPLES = 50
KAPPA = 2.0
N_CANDIDATES = 20000

# Train the model
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

losses = []
model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    pred = model(X_tensor)
    loss = criterion(pred, y_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.6f}")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (normalized)')
plt.title('Neural Network Training Loss')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Final training metrics
model.eval()
with torch.no_grad():
    train_pred = model(X_tensor).squeeze().numpy()
train_pred_orig = train_pred * y_std + y_mean
train_r2 = 1 - np.sum((y_w5 - train_pred_orig)**2) / np.sum((y_w5 - y_w5.mean())**2)
print(f"\nTraining R²: {train_r2:.6f}")

### Step 3: UCB Acquisition with MC Dropout

Run 50 stochastic forward passes (dropout enabled) for each candidate point:
- **mu(x)** = mean of MC predictions
- **sigma(x)** = std of MC predictions
- **UCB(x) = mu(x) + kappa * sigma(x)** where kappa = 2.0

In [ ]:
# Generate random candidate points
np.random.seed(42)
candidates = np.random.uniform(0, 0.999999, size=(N_CANDIDATES, 5))

# Normalize candidates
cand_norm = (candidates - X_mean) / X_std
cand_tensor = torch.tensor(cand_norm, dtype=torch.float32)

# MC Dropout: Run model in train mode for stochastic forward passes
model.train()  # Enable dropout
mc_predictions = []
with torch.no_grad():
    for _ in range(MC_SAMPLES):
        pred = model(cand_tensor).squeeze().numpy()
        mc_predictions.append(pred)

mc_predictions = np.array(mc_predictions)  # shape: (MC_SAMPLES, N_CANDIDATES)

# Denormalize predictions
mc_preds_orig = mc_predictions * y_std + y_mean

# Compute mean and uncertainty
mu = mc_preds_orig.mean(axis=0)
sigma = mc_preds_orig.std(axis=0)

# UCB acquisition
ucb = mu + KAPPA * sigma

# Select best candidate
best_idx = np.argmax(ucb)
next_point_w5 = np.clip(candidates[best_idx], 0.0, 0.999999)

print("UCB Acquisition Results (MC Dropout):")
print(f"  Best UCB value:     {ucb[best_idx]:.6f}")
print(f"  MC mean prediction: {mu[best_idx]:.6f}")
print(f"  MC std (sigma):     {sigma[best_idx]:.6f}")
print(f"  Next sample point:  {next_point_w5}")
print()
print("MC prediction statistics:")
print(f"  Mean of mu:    {mu.mean():.6f}")
print(f"  Max of mu:     {mu.max():.6f}")
print(f"  Mean of sigma: {sigma.mean():.6f}")
print(f"  Max of sigma:  {sigma.max():.6f}")

### Step 4: Feature Importance via Input Gradients

Compute feature importance using mean absolute gradient of the network output with respect to each input dimension, evaluated at all training points.

In [ ]:
# Feature importance via input gradient magnitude
model.eval()
X_grad = torch.tensor(X_norm, dtype=torch.float32, requires_grad=True)
output = model(X_grad).sum()
output.backward()

# Mean absolute gradient per dimension
grad_importance = X_grad.grad.abs().mean(dim=0).numpy()
grad_importance = grad_importance / grad_importance.sum()  # Normalize to sum to 1

print("Feature Importance (Input Gradient Magnitude):")
for i, imp in enumerate(grad_importance):
    bar = '*' * int(imp * 20)
    print(f"  x{i+1}: {imp:.4f} ({bar})")

### Step 5: Visualize Neural Network Surrogate

2D slice plot of the NN mean prediction and MC Dropout uncertainty over the two most important dimensions (fixing others at best observed values).

In [ ]:
# Top 2 dimensions by gradient importance
top2 = np.argsort(grad_importance)[-2:]
fixed_dims = [d for d in range(5) if d not in top2]
best_point = X_w5[y_w5.argmax()]

print(f"Top 2 dimensions: x{top2[0]+1} (imp={grad_importance[top2[0]]:.4f}), x{top2[1]+1} (imp={grad_importance[top2[1]]:.4f})")
print(f"Fixed dimensions: " + ", ".join([f"x{d+1}={best_point[d]:.4f}" for d in fixed_dims]))

# Create 2D grid
n_grid = 50
d0_grid = np.linspace(0, 1, n_grid)
d1_grid = np.linspace(0, 1, n_grid)
D0, D1 = np.meshgrid(d0_grid, d1_grid)

# Build full grid points
grid_points = np.zeros((n_grid * n_grid, 5))
grid_points[:, top2[0]] = D0.ravel()
grid_points[:, top2[1]] = D1.ravel()
for d in fixed_dims:
    grid_points[:, d] = best_point[d]

# Normalize and predict with MC Dropout
grid_norm = (grid_points - X_mean) / X_std
grid_tensor = torch.tensor(grid_norm, dtype=torch.float32)

model.train()  # Enable dropout for MC
grid_mc = []
with torch.no_grad():
    for _ in range(MC_SAMPLES):
        pred = model(grid_tensor).squeeze().numpy()
        grid_mc.append(pred)
grid_mc = np.array(grid_mc) * y_std + y_mean

grid_mu = grid_mc.mean(axis=0).reshape(n_grid, n_grid)
grid_sigma = grid_mc.std(axis=0).reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: NN mean prediction
ax1 = axes[0]
c1 = ax1.contourf(D0, D1, grid_mu, levels=30, cmap='viridis')
ax1.scatter(X_w5[:, top2[0]], X_w5[:, top2[1]], c='red', edgecolors='white', s=60, zorder=5, label='Observed')
ax1.scatter(next_point_w5[top2[0]], next_point_w5[top2[1]], c='yellow', marker='*', s=200, edgecolors='black', zorder=6, label='Next point')
fixed_str = ", ".join([f"x{d+1}={best_point[d]:.2f}" for d in fixed_dims])
ax1.set_xlabel(f'x{top2[0]+1}'); ax1.set_ylabel(f'x{top2[1]+1}')
ax1.set_title(f'NN Mean ({fixed_str})')
ax1.legend(loc='lower right', fontsize=8)
plt.colorbar(c1, ax=ax1)

# Plot 2: MC Dropout uncertainty
ax2 = axes[1]
c2 = ax2.contourf(D0, D1, grid_sigma, levels=30, cmap='YlOrRd')
ax2.scatter(X_w5[:, top2[0]], X_w5[:, top2[1]], c='blue', edgecolors='white', s=60, zorder=5, label='Observed')
ax2.set_xlabel(f'x{top2[0]+1}'); ax2.set_ylabel(f'x{top2[1]+1}')
ax2.set_title(f'MC Dropout Uncertainty ({fixed_str})')
ax2.legend(loc='lower right', fontsize=8)
plt.colorbar(c2, ax=ax2)

# Plot 3: Feature importance
ax3 = axes[2]
dims = [f'x{i+1}' for i in range(5)]
colors = ['steelblue' if i in top2 else 'lightgray' for i in range(5)]
ax3.barh(dims, grad_importance, color=colors)
ax3.set_xlabel('Importance')
ax3.set_title('Feature Importance (Gradient)')
ax3.set_xlim(0, max(grad_importance) * 1.2)

plt.tight_layout()
plt.show()

### Step 6: Convergence Plot

Running maximum (best observed value) across all observations.

In [ ]:
# Convergence plot
running_max = np.maximum.accumulate(y_w5)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(y_w5) + 1), running_max, 'b-o', markersize=6, label='Running Maximum')
plt.scatter(range(1, len(y_w5) + 1), y_w5, c='gray', alpha=0.5, s=30, label='Individual Observations')
plt.axvline(x=15.5, color='red', linestyle='--', alpha=0.5, label='Initial -> Weekly boundary')
plt.xlabel('Observation Number')
plt.ylabel('Objective Value')
plt.title('Function 6 - Convergence Plot (Week 5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best observed value: {y_w5.max():.6f}")
print(f"Achieved at observation: {y_w5.argmax() + 1}")

### Step 7: Format Submission Query

Format the proposed next sample point as `x1-x2-x3-x4-x5` with 6 decimal places.

In [ ]:
# Format submission query
def format_query(point):
    clamped = [max(0.0, min(0.999999, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

submission_query_w5 = format_query(next_point_w5)

print("=" * 60)
print("WEEK 5 SUBMISSION QUERY FOR FUNCTION 6")
print("=" * 60)
print(f"Surrogate: Neural Network ({model.__class__.__name__})")
print(f"Architecture: 5 -> 64 -> 32 -> 1")
print(f"Dropout: 0.2, MC samples: {MC_SAMPLES}")
print(f"Acquisition: UCB (kappa={KAPPA})")
print(f"Training R²: {train_r2:.6f}")
print(f"Next point: {next_point_w5}")
print(f"MC mean: {mu[best_idx]:.6f}")
print(f"MC std:  {sigma[best_idx]:.6f}")
print(f"")
print(f">>> SUBMISSION: {submission_query_w5}")
print("=" * 60)

### Model Comparison

**Neural Network vs GP (Initial Section):**
- GP uses a Matern 5/2 kernel with Expected Improvement and ARD lengthscales for 5D — provides principled posterior uncertainty but cubic scaling with data points.
- The NN uses MC Dropout for uncertainty — each stochastic forward pass samples a different subnetwork, and the variance across passes estimates epistemic uncertainty.
- For f6's 5D problem with negative objective values, the NN can learn complex non-linear boundaries without the GP's smoothness assumptions.
- Input gradient magnitude provides interpretable feature importance, analogous to GP's inverse lengthscales.
- Key trade-off: GP posterior is exact given the kernel; NN+MC Dropout uncertainty is approximate but scales better to higher dimensions.